# Whats is a Domain Data Componet? 

It is a encapsulation of heterogenoeus domain data (logs, tables, etc). It might have some processing 
capabilities itself that can be used by client code. 

In every case, the different domain data components derive from a base class BaseDataComponent

They must implement the method set_data( data : Any )

Tools (as we will later) can become stateful by accessing this data component


In [ ]:
from abc import ABC, abstractmethod
from typing import Any
import pandas  as pd 
from typing_extensions import Self

class BaseDataComponent(ABC):
    """
    Base class for stateful domain data components.

    Concrete implementations define how raw domain data is stored
    and exposed to the rest of the system.
    """

    def __init__(self):
        self._data = None
        self._metadata = None 

    @abstractmethod
    def set_data(self, *args, **kwargs) -> Self:
        """
        Set or replace the domain data.
        """
        raise NotImplementedError

    def _check_if_data(self) -> None:
        """
        Raise an error if data has not been set.
        """
        if self._data is None:
            raise RuntimeError(
                "No data has been set. Call set_data(...) first."
            )

        

#### Example of a particular implementation for CRMResultsDataComponent 

In [ ]:


class CRMResultsData(BaseDataComponent):

    def set_data(self, crm_results: Any, metadata: dict[str, Any] | None = None ) -> Self:

        self._data =   crm_results
        self._metadata = metadata
        self._state = 123 
        return self 

    def internal_method(self):
        print("I do something to my internal data. But now will just return a number which corresponds to my state")
        return self._state 

        
project_name = xx
study_name   = xx 

raw_crm_results = { 'simulation_config':'some json/dict',
'liquid_history_match':pd.DataFrame({}),
'connectivity_data': [1,2,3],
'water_history_match': pd.DataFrame({}),
'etc':'whatever needed'
}

# read the data we will operate on, or provide lambda functions to read it on-the-fly as needed 
crm_results_data_component = CRMResultsData( ).set_data( raw_crm_results )




In [35]:
crm_results_data_component._data

{'simulation_config': 'some json/dict',
 'liquid_history_match': Empty DataFrame
 Columns: []
 Index: [],
 'connectivity_data': [1, 2, 3],
 'water_history_match': Empty DataFrame
 Columns: []
 Index: [],
 'etc': 'whatever needed'}

So ... read the data needed to work with the component, store it.

# Whats are Domain  Tools ? 

It is a encapsulation of functions (or methods) that operate on data and that can be used by agents

Tools are stateful: this means that they can hold memory to whatever data was available or created along the execution of a turn. 
This way, Tools can operate on massive datasets without needed to pass data directly to the llms.

There is a key advantage of this setup: The data can change ( the user selects another dataset, project, etc ) but the tools do not 
need to be regenerated or passed to an llm. They just "know" what data state is current via an internal pointer.
The user can change projects, simulations, etc. The system becomes immediately aware after set_data call to the respective 
domain data component

holds an optional _data_component reference
- set_data_component(...)
- _check_if_data_component(...)
- get_agent_tools(...)



In [ ]:
import inspect

from langchain_core.tools import StructuredTool


class BaseDomainTools:
    """
    Base class for domain-specific tools exposed to an LLM agent.

    Concrete subclasses define public methods with docstrings.
    Those methods are automatically exposed as StructuredTools.
    """

    def __init__(self):
        self._data_component = None

    def set_data_component(self, data_component: BaseDataComponent) -> None:
        """
        Attach the data component used by these tools.
        """
        self._data_component = data_component

    def _check_if_data_component(self) -> None:
        """
        Return the attached data component or raise if none is available.
        """
        if self._data_component is None:
            raise RuntimeError(
                "No data component has been assigned. "
                "Call set_data_component(...) first."
            )

        return None 

    def get_agent_tools(self) -> list[StructuredTool]:
        """
        Return the public documented methods exposed to the LLM agent.
        """
        tools = []

        excluded_methods = {
            "set_data_component",
            "get_agent_tools",
        }

        for name in dir(self):
            if name.startswith("_") or name in excluded_methods:
                continue

            attr = getattr(self, name)

            if not callable(attr):
                continue

            doc = inspect.getdoc(attr)

            if not doc:
                continue

            tools.append(
                StructuredTool.from_function(
                    func=attr,
                    name=name,
                    description=doc,
                )
            )

        return tools

    

#### Example of a particular implementation for CRMResultsToolsComponent 

In [ ]:
from typing import cast
import inspect


class CRMResultsTools( BaseDomainTools ):

    def __init__(self):
        super().__init__( )
        self._state = {} 


    @property
    def domain_data(self) -> CRMResultsData:
        """
        Return the attached data component wrapper (data + some methods) with its concrete type.
        """
        wrapper = cast(CRMResultsData,self._data_component)
        return wrapper
    
    @property
    def raw_data(self) -> Any:
        """
        Return the raw data (arrays, dataframes, etc )
        """
        wrapper = cast(CRMResultsDataComponent,self._data_component)
        return self.domain_data._data

    

    def sum_all_connectivities(self)->float:
        """
        Returns the sum of the connectivities. 
        """
        _ = self._check_if_data_component()

        _data = self.raw_data   
        print(_data['connectivity_data'])
        result = sum(_data['connectivity_data']) 

        print("sum of connectivity is", result )
        return result 

        
    def projectivity( self ):
        """
        Compute the projectivity of the data, which is just a function of the current state.
        Agents can use this tool when estimating the projectivity in the data
        """

        #raises an exception if called without setting the data first  
        _ = self._check_if_data_component()

    
        projectivity = self.domain_data.internal_method() * 2 
        return f"the projectivity is: {projectivity}"


crm_results_tools = CRMResultsTools()
crm_results_tools.set_data_component( crm_results_data_component ) 

result1  = crm_results_tools.projectivity() 
print( result1 )


print("sum of connectivities is ")
crm_results_tools.sum_all_connectivities() 

# now the user selects a new project, or model...everything changed. Lets simulate that
raw_crm_results2 = { 'simulation_config':'some json/dict',
'liquid_history_match':pd.DataFrame({}),
'connectivity_data': [111,2222,3333,444,555 ],
'water_history_match': pd.DataFrame({}),
'etc':'whatever needed'
}
crm_results_data_component.set_data( raw_crm_results2 )


print("sum of connectivities after new data arrived (new project, new selection in the UI) is ")
crm_results_tools.sum_all_connectivities() 


I do something to my internal data. But now will just return a number which corresponds to my state
the projectivity is: 246
sum of connectivities is 
[1, 2, 3]
sum of connectivity is 6
sum of connectivities after new data arrived (new project, new selection in the UI) is 
[111, 2222, 3333, 444, 555]
sum of connectivity is 6665


6665

We can have many of these, which would be useful as "standalone" but also could be 
used as "tools" for a more general agent


In [ ]:


class CRMResultsPlanner: # type: ignore
    #pointer to data if needed and whatever tools  
    pass 

    def run( self, query:str ):
        pass 
    

class CRMResultsReportGenerator:
    #pointer to data if needed and whatever tools  
    pass 

    def run( self, query:str ):
        pass 


class CRMResultsOpportunityScanner:
    def __init__(self):
        self.description = "use this component to identify chanelling"
        #pointer to data if needed and whatever tools  

    def run( self, query:str ):
        pass 



class CRMResultsEventsAnalysis:

    def __init__(self):
        self.description = "use this component to identify chanelling"
        


class SmartCRMResultsSystem:

    def __init__(self):

        AComponentAndItsTools1: CRMResultsPlanner

        AComponentAndItsTools2: CRMResultsOpportunityScanner

        AComponentAndItsTools3: CRMResultsEventsAnalysis     

        def run( self, query:str ):

            # plan = generate a plan, account for history of conversation 
            # call the agents as per plan
            # collect results
            # return results in a format that the presenter can understand 
            # 
            #     

<class '__main__.CRMResultsDataComponent'>
I do something to my internal data. But now will just return a number which corresponds to my state


'the projectivity is: 246'

# Prototype 1.
Report generation

Lets start with the data. For now, lets assume that we dont have a "sql-like" analyst and the data is just a table of lambdas, taus, etc...
We also will use a simple ReAct agent off-the-shelve and we will require a planning step.



Later to do: 
change _data to _raw_data in the base class.
Do the propper semantic model using pydantic objects with validation 
The base class tools should receive the llm ? think .

In [179]:
import pandas as pd
from pathlib import Path
import pprint 
import os,sys
from typing import cast
import inspect
from dataclasses import dataclass

sys.path.append(".")
sys.path.append("..//")
sys.path.append("..//..//")
from langgraph.graph import END, StateGraph
from langchain_core.tools import StructuredTool, Tool
from langchain.agents import create_agent


from visualization_system.common.base_domain_tools import BaseDataComponent, BaseDomainTools 
from visualization_system.common.get_llm import azure_llm_if

get_llm = azure_llm_if


In [194]:

@dataclass 
class CRMResultsToolsConfiguration:

    crm_results_report_prompt = """ 
    You act as a Reservoir Engineer specialized in Waterflood modelling and Capacitance 
    Resistance Models (CRM).
 
You are provided with one summary results table: "CRM parameter results" that summarizes results of a history match simulation. The simulator 
aims at fitting liquid production time series observed in producers with a CRM-P model. The parameters found 
by the simulator are the GAINS, TAU, TAUP, PRODUCTIVITY described below. 

You are also given contextual information about the table

Your job is to aid in the interpretation of results while answering user questions.
You must use the results provided, own knowledge and context provided to analyze the results.
you shoud be able to produce a concise report summarizing simulation results and go into the details when asked to.
 
Important:
Be precise and concise. Avoid long responses unless the user asks for a detailed explanation.

{table_data}

{context}


"""
        

In [ ]:
class CRMResultsData(BaseDataComponent):

    def set_data(self, crm_results: Any, metadata: dict[str, Any] | None = None ) -> Self:

        self._data     = crm_results
        self._metadata = metadata
        self._state = 123 
        return self 

    def __init__( self ):
        self.context = {'summary': """
Table description 												
The table shows simulation results for a history match of liquid production for a number of producers. 
Columns: 		
-Injector: Injector well  name (source)														
-Producer: producer well name  (sink)														
-GAIN: The fraction of the water injected in an injector that is recovered as liquid in a connected producer, denoted as f_{ij}. This is the "gain" of production due to injection														
-TAU:  Characteristic response time. This is the time the information takes to propagate from injector to producer 														
-TAUP: Characteristic time for the depletion in the producer well
-PRODUCTIVITY: Proportionality coefficient between liquid production rate and pressure changes. 
It depends on the producer itself and the flowing bottom hole pressure in the producer and not on 
its connections to injectors, gains or injectors themselves  

-LO: Adjustment coefficient for primary production. Ignore it.  
-QUALITY_SCORE: Derived metric, Maximum value = 1, minimum value = 0. The higher the value the better the history match obtained by the simulator for the specific producer.
The quality score is computed as a combination of BIAS_RATIO, CORRELATION and VARIANCE_RATIO
-CORRELATION: measures the correlation between the real production in time (signal) and the simulator prediction. Values < 0.5 usually indicate a poor match
-VARIANCE_RATIO: shows the ratio of the variance between the simulator prediction time series and the observed one. 


Facts:
-An injector injects water 
-A producer, produces water oil and gas. The liquid production is the sum of oil + water + gas 
-The sum over the producer gains for each injector needs to add up to less than 1. namely sum_p { f_{ip}} < 1 for all i.

Concepts:
-A producer is supported when one or more injectors support the producer. This is evidenced in production gains greater than zero from 1+ injectors connected to the producer  
-An injector supports one or more producers when there is a non negligible gain in one or more producers due to that injector  
-A high-utility injector is one that supports one or more producers, and for which sum_p {f_ip} is close to 1.
-Channeling: A potential high-permeability between a pair of connected producer-injector. It might be due to fracturing, in which case TAU is generally small. 

Physical meanings:

Large values of TAUP:
usually indicate a slow (flat) production decline even in the absence of injection support. 
This can happen due to sources other than injectors contributing to production, such as aquifers. It might also indicate that 
the data on production and injection shows little variance and the simulator fits the nearly flat signals by choosing 
large values of TAUP.

Large values of TAU: In general, they may indicate a similar situation as large values of TAUP. 

Small values of TAU: Indicate a short response time between a change in injection and the observed change in production 
for injector-produer pairs. If there is ALSO a significant GAIN (close to 1), both can indicate a potential "channeling"

Large GAINS (close to 1): Indicate a good connectivity between the injector and producer.
Small GAINS (less than 0.1): Indicate a poor connectivity between the injector and producer.
 
LOW PRODUCTIVITY: PRODUCTIVITY values of zero across all the rows usually indicate that the simulation was done 
without infnformation on bottom-hole pressure. Small values (but not zero) indicate that production is due to 
connected injectors and depletion. PRODUCTIVITY of the order of 1 indicates that production is driven by pressure 
controls at the surface and potentially less by waterflooding (injection)

HIGH Gain and low TAU: This combination indicates a strong and fast response between the injector and producer. It might indicate a potential channeling between the injector and producer
which might indicate hydraulic fracturing. In such cases, it is worth checking the injection history 
of the given injector in relation to typical values observed in the field. 
If the injection history shows relatively high injection rates (above field mean), 
then the combination of HIGH GAIN (>0.7) and low TAU ( < 2 ) might indicate a potential channeling 
between the injector and producer. Other diagnostic plots such as the Hall plot can be used. 
""" 
                    }




class CRMResultsTools( BaseDomainTools ):

    def __init__(self, llm, configuration:Any|None = None ):
        super().__init__( )

        self.llm = llm 
        self._state = {} 
        self._config = configuration or CRMResultsToolsConfiguration()


    @property
    def domain_data(self) -> CRMResultsData:
        """
        Return the attached data component wrapper (data + some methods) with its concrete type.
        """
        wrapper = cast(CRMResultsData,self._data_component)
        return wrapper
    
    @property
    def raw_data(self) -> Any:
        """
        Return the raw data (arrays, dataframes, etc )
        """
        wrapper = cast(CRMResultsData,self._data_component)
        return self.domain_data._data

    def generate_report(self):
        """
        Returns a textual summary of simulation results.
        The report includes an assessment of quality, a list of stranded injectors (if any), 
        A list of unsupported producers, and potential injector-producer chanelling scenarios  
        
        Before a conclusion on chanelling can be drawn, clients must collect more evidence that support chanelling. 
        For instance, statistics on the history of injection of the relevant injector 
        relative to field common values, the presence of faults aligned with the injector-producer 
        pair can be used to support the hypothesis of chanelling.   
        """

        prompt_template = self._config.crm_results_report_prompt 

        (df, metadata) = self.domain_data._data['summary_results_table'].to_dict( orient='records' ),self.domain_data._metadata 

          


        inputs = {'table_data':df,
                   'context': metadata
        }

        prompt = prompt_template.format( **inputs )

        response = self.llm.invoke( prompt )
        

        return response, "All the simulations ran just fine. " 

    def check_injectors_balanced(self):
        return "Yes, they are all balanced"
    
    def record_plan( self, plan ):
        print("Recording the plan")
        print( plan )
        return "plan recorded OK"


class CRMResultsAnalyst:

    def __init__(self, llm ,data_component:CRMResultsData|None=None,
                 domain_tools:CRMResultsTools|None=None ):

        self.llm = llm 
        self.data_component = data_component or CRMResultsData()
        self.domain_tools   = domain_tools or CRMResultsTools( llm )
        self.domain_tools.set_data_component( self.data_component )


    def set_raw_data( self, raw_data,metadata):
        self.data_component.set_data( raw_data, metadata )


    def run( self, query:str, background:str|None = None )->Any:
        message =  "[run] I am a hard-coded runner for CRMResultsAnalyst that creates an agent inplace"
        print(message)

        agent_tools = self.domain_tools.get_agent_tools()
        prompt = "You help with questions related to CRM results"

        agent = create_agent(model = self.llm,
                             system_prompt= prompt,
                             tools=agent_tools
                             )

      
        response = agent.invoke({"messages": [{"role": "user", "content": query}]})


        return response  

        # need to generate a basic plan 
    


    
# this is just a bunch of backend helpers. Nthing fancy      
def check_if_need_refresh_data( project_name:str, study_name:str):
    print("[check_if_need_refresh_data] is hard-coded")
    return True

def load_results_data( project_name, study_name) :         
    print("[load_results_data] is hard-coded")

    DATAPATH = Path("../datasets/")
    FILE = "summary_results1.csv"
    df = pd.read_csv( DATAPATH/FILE )
    semantic_model = {'summary_results_table': """
    Table description 												
    The table shows simulation results for a history match of liquid production for a number 
    of producers. 
    Columns: 		
    -Injector: Injector well  name (source)														
    -Producer: producer well name  (sink)														
    -GAIN: The fraction of the water injected in an injector that is recovered as liquid in a connected producer, denoted as f_{ij}. This is the "gain" of production due to injection														
    -TAU:  Characteristic response time. This is the time the information takes to propagate from injector to producer 														
    -TAUP: Characteristic time for the depletion in the producer well
    -PRODUCTIVITY: Proportionality coefficient between liquid production rate and pressure changes. 
    It depends on the producer itself and the flowing bottom hole pressure in the producer and not on 
    its connections to injectors, gains or injectors themselves  

    -LO: Adjustment coefficient for primary production. Ignore it.  
    -QUALITY_SCORE: Derived metric, Maximum value = 1, minimum value = 0. The higher the value the better the history match obtained by the simulator for the specific producer.
    The quality score is computed as a combination of BIAS_RATIO, CORRELATION and VARIANCE_RATIO
    -CORRELATION: measures the correlation between the real production in time (signal) and the simulator prediction. Values < 0.5 usually indicate a poor match
    -VARIANCE_RATIO: shows the ratio of the variance between the simulator prediction time series and the observed one. 


    Facts:
    -An injector injects water 
    -A producer, produces water oil and gas. The liquid production is the sum of oil + water + gas 
    -The sum over the producer gains for each injector needs to add up to less than 1. namely sum_p { f_{ip}} < 1 for all i.

    Concepts:
    -A producer is supported when one or more injectors support the producer. This is evidenced in production gains greater than zero from 1+ injectors connected to the producer  
    -An injector supports one or more producers when there is a non negligible gain in one or more producers due to that injector  
    -A high-utility injector is one that supports one or more producers, and for which sum_p {f_ip} is close to 1.
    -Channeling: A potential high-permeability between a pair of connected producer-injector. It might be due to fracturing, in which case TAU is generally small. 

    Physical meanings:

    Large values of TAUP:
    usually indicate a slow (flat) production decline even in the absence of injection support. 
    This can happen due to sources other than injectors contributing to production, such as aquifers. It might also indicate that 
    the data on production and injection shows little variance and the simulator fits the nearly flat signals by choosing 
    large values of TAUP.

    Large values of TAU: In general, they may indicate a similar situation as large values of TAUP. 

    Small values of TAU: Indicate a short response time between a change in injection and the observed change in production 
    for injector-produer pairs. If there is ALSO a significant GAIN (close to 1), both can indicate a potential "channeling"

    Large GAINS (close to 1): Indicate a good connectivity between the injector and producer.
    Small GAINS (less than 0.1): Indicate a poor connectivity between the injector and producer.
    
    LOW PRODUCTIVITY: PRODUCTIVITY values of zero across all the rows usually indicate that the simulation was done 
    without infnformation on bottom-hole pressure. Small values (but not zero) indicate that production is due to 
    connected injectors and depletion. PRODUCTIVITY of the order of 1 indicates that production is driven by pressure 
    controls at the surface and potentially less by waterflooding (injection)

    HIGH Gain and low TAU: This combination indicates a strong and fast response between the injector and producer. It might indicate a potential channeling between the injector and producer
    which might indicate hydraulic fracturing. In such cases, it is worth checking the injection history 
    of the given injector in relation to typical values observed in the field. 
    If the injection history shows relatively high injection rates (above field mean), 
    then the combination of HIGH GAIN (>0.7) and low TAU ( < 2 ) might indicate a potential channeling 
    between the injector and producer. Other diagnostic plots such as the Hall plot can be used. 
    """ 
                        }

    return {'summary_results_table':df}, semantic_model

def preprocess_query( query:str)->str:
    return query   

1. The app initializes the data object and the tools.
We dont need any data yet. 

In [208]:
# We will first use the analyst standalone and then as a tool for a coordinator architecture  

llm = get_llm()
crm_analyst = CRMResultsAnalyst( llm )


zero temp, seed 42, top_p = 1


2. A query arrives from the UI

In [209]:
payload = {
    'query':'produce a summary report',
    'project_name': 'Any project name',
    'study_name':'Any study name'
}

project_name, study_name = payload['project_name'], payload['study_name']

project_name, study_name 

('Any project name', 'Any study name')

3. We check if the project or study name changed. 
In which case, we need to reload the data 

Yet everything is already hooked. Just change the data and it should work

In [211]:
if check_if_need_refresh_data( project_name, study_name):
    data, semantic_model = load_results_data(project_name, study_name)
    crm_analyst.set_raw_data( data, semantic_model )

# simple chekcs 
#display(crm_analyst.data_component._data['summary_results_table'].head(5))
#pprint.pprint( crm_analyst.data_component._metadata['summary_results_table'])

#data  = crm_analyst.data_component
tools = crm_analyst.domain_tools 

response, _ = tools.generate_report()

#query = preprocess_query( payload['query'])
#response = crm_analyst.run( query )
#pprint.pprint( response['messages'][-1].content )




[check_if_need_refresh_data] is hard-coded
[load_results_data] is hard-coded


In [220]:
import markdown 
from IPython.display import display, Markdown

display(Markdown(response.content))

#html_output = markdown.markdown(response.content)
#print(html_output)


### Summary Report: CRM-P Simulation Results

#### Overview:
The provided table summarizes the results of a history match simulation using a Capacitance Resistance Model with Productivity (CRM-P). The goal of the simulation is to fit liquid production time series observed in producers based on water injection from injectors. Key parameters include **GAIN**, **TAU**, **TAUP**, **PRODUCTIVITY**, and **QUALITY_SCORE**, among others. Below is a concise interpretation of the results.

---

### Key Observations:

1. **General Trends**:
   - **PRODUCTIVITY**: All values are either zero or extremely small, indicating that the simulation was performed without bottom-hole pressure (BHP) data. Production is therefore modeled as being driven by injection and depletion rather than pressure controls.
   - **GAIN**: Gains vary significantly across injector-producer pairs, with some close to 1 (indicating strong connectivity) and others near zero (indicating poor connectivity).
   - **TAU**: Values range from very small (e.g., 0.5) to large (e.g., 32.6), reflecting varying response times between injectors and producers.
   - **QUALITY_SCORE**: Scores range from negative values (poor match) to values close to 1 (excellent match). High-quality matches are associated with high **CORRELATION** and **VARIANCE_RATIO**.

2. **High-Utility Injectors**:
   - **MG-0380_I**: Supports multiple producers (e.g., MG-0008_P, MG-0068_P, MG-0278_P) with varying gains. For MG-0278_P, the gain is very high (0.997), indicating strong connectivity.
   - **MG-0526_I**: Supports producers like MG-0553_P with a high gain (0.997), suggesting it is a high-utility injector.

3. **Channeling Indicators**:
   - **MG-0380_I → MG-0278_P**: High **GAIN** (0.997) and low **TAU** (13.2) suggest strong connectivity and potential channeling.
   - **MG-0621_I → MG-0109_P**: High **GAIN** (0.997) and low **TAU** (0.59) suggest a strong and fast response, potentially indicating channeling.

4. **Poor Connectivity**:
   - Several injector-producer pairs have negligible **GAIN** values (e.g., MG-0380_I → MG-0008_P with GAIN = 6.13e-06), indicating poor connectivity or no significant support.

5. **High-Quality Matches**:
   - **MG-0289_I → MG-0567_P**: High **QUALITY_SCORE** (0.981), **CORRELATION** (0.976), and **GAIN** (0.578) indicate a strong match and good connectivity.
   - **MG-0528_I → MG-0579_P**: High **QUALITY_SCORE** (0.978) and **CORRELATION** (0.979) suggest a strong match.

6. **Low-Quality Matches**:
   - **MG-0270_I → MG-0058_P**: Poor **QUALITY_SCORE** (-0.078), negative **CORRELATION** (-0.203), and low **GAIN** (0.005) indicate a poor match and weak connectivity.

---

### Detailed Insights:

1. **Channeling Potential**:
   - **MG-0380_I → MG-0278_P**: High **GAIN** (0.997) and low **TAU** (13.2) suggest potential channeling. Further diagnostics (e.g., Hall plots) and injection history analysis are recommended to confirm.
   - **MG-0621_I → MG-0109_P**: Similar indicators (high **GAIN** and low **TAU**) suggest possible channeling.

2. **Slow Decline Producers**:
   - **MG-0109_P**: Large **TAUP** (50.0) suggests a slow production decline, potentially due to aquifer support or flat production data.

3. **Injector-Producer Connectivity**:
   - **MG-0380_I**: Supports multiple producers with varying gains. Strong connectivity is observed with MG-0278_P (**GAIN** = 0.997).
   - **MG-0526_I**: High **GAIN** with MG-0553_P (**GAIN** = 0.997) indicates strong connectivity.

4. **Producers with High-Quality Matches**:
   - **MG-0567_P**: Supported by MG-0289_I with a high **GAIN** (0.578) and excellent **QUALITY_SCORE** (0.981).
   - **MG-0579_P**: Supported by MG-0528_I with a high **QUALITY_SCORE** (0.978).

5. **Producers with Poor Matches**:
   - **MG-0058_P**: Poor match with MG-0270_I (**QUALITY_SCORE** = -0.078, **CORRELATION** = -0.203).

---

### Recommendations:

1. **Channeling Investigation**:
   - Focus on injector-producer pairs with high **GAIN** and low **TAU** (e.g., MG-0380_I → MG-0278_P, MG-0621_I → MG-0109_P). Analyze injection history and perform additional diagnostics (e.g., Hall plots).

2. **Improve History Match**:
   - For pairs with low **QUALITY_SCORE** (e.g., MG-0270_I → MG-0058_P), consider revisiting the input data, including injection/production rates and model assumptions.

3. **High-Utility Injectors**:
   - Prioritize injectors like MG-0380_I and MG-0526_I for operational optimization, as they support multiple producers with high gains.

4. **Further Analysis**:
   - Investigate producers with large **TAUP** values (e.g., MG-0109_P) to understand the source of slow production decline.

---

Let me know if you need further details or specific analyses!

In [ ]:
DATAPATH = Path("../datasets/")
FILE = "summary_results1.csv"
df = pd.read_csv( DATAPATH/FILE )

data = {'summary': df}
semantic_models = { 'summary': """Table description 												
The table shows simulation results for a history match of liquid production for a number of producers. 
Columns: 		
-Injector: Injector well  name (source)														
-Producer: producer well name  (sink)														
-GAIN: The fraction of the water injected in an injector that is recovered as liquid in a connected producer, denoted as f_{ij}. This is the "gain" of production due to injection														
-TAU:  Characteristic response time. This is the time the information takes to propagate from injector to producer 														
-TAUP: Characteristic time for the depletion in the producer well
-PRODUCTIVITY: Proportionality coefficient between liquid production rate and pressure changes. 
It depends on the producer itself and the flowing bottom hole pressure in the producer and not on 
its connections to injectors, gains or injectors themselves  

-LO: Adjustment coefficient for primary production. Ignore it.  
-QUALITY_SCORE: Derived metric, Maximum value = 1, minimum value = 0. The higher the value the better the history match obtained by the simulator for the specific producer.
The quality score is computed as a combination of BIAS_RATIO, CORRELATION and VARIANCE_RATIO
-CORRELATION: measures the correlation between the real production in time (signal) and the simulator prediction. Values < 0.5 usually indicate a poor match
-VARIANCE_RATIO: shows the ratio of the variance between the simulator prediction time series and the observed one. 


Facts:
-An injector injects water 
-A producer, produces water oil and gas. The liquid production is the sum of oil + water + gas 
-The sum over the producer gains for each injector needs to add up to less than 1. namely sum_p { f_{ip}} < 1 for all i.

-QUALITY_SCORE: Derived metric, Maximum value = 1, minimum value = 0. The higher the value the better the history match obtained by the simulator for the specific producer.
The quality score is computed as a combination of BIAS_RATIO, CORRELATION and VARIANCE_RATIO
-CORRELATION: measures the correlation between the real production in time (signal) and the simulator prediction. Values < 0.5 usually indicate a poor match
-VARIANCE_RATIO: shows the ratio of the variance between the simulator prediction time series and the observed one. 
-Channeling: A potential high-permeability between a pair of connected producer-injector. It might be due to fracturing, in which case TAU is generally small. 

Physical meanings:

Large values of TAUP:
usually indicate a slow (flat) production decline even in the absence of injection support. 
This can happen due to sources other than injectors contributing to production, such as aquifers. It might also indicate that 
the data on production and injection shows little variance and the simulator fits the nearly flat signals by choosing 
large values of TAUP.

Large values of TAU: In general, they may indicate a similar situation as large values of TAUP. 

Small values of TAU: Indicate a short response time between a change in injection and the observed change in production 
for injector-produer pairs. If there is ALSO a significant GAIN (close to 1), both can indicate a potential "channeling"

Large GAINS (close to 1): Indicate a good connectivity between the injector and producer.
Small GAINS (less than 0.1): Indicate a poor connectivity between the injector and producer.
 
LOW PRODUCTIVITY: PRODUCTIVITY values of zero across all the rows usually indicate that the simulation was done 
without infnformation on bottom-hole pressure. Small values (but not zero) indicate that production is due to 
connected injectors and depletion. PRODUCTIVITY of the order of 1 indicates that production is driven by pressure 
controls at the surface and potentially less by waterflooding (injection)

HIGH Gain and low TAU: This combination indicates a strong and fast response between the injector and producer. It might indicate a potential channeling between the injector and producer
which might indicate hydraulic fracturing. In such cases, it is worth checking the injection history 
of the given injector in relation to typical values observed in the field. 
If the injection history shows relatively high injection rates (above field mean), 
then the combination of HIGH GAIN (>0.7) and low TAU ( < 2 ) might indicate a potential channeling 
between the injector and producer. Other diagnostic plots such as the Hall plot can be used. 
"""
}


In [188]:



data_component  = CRMResultsData()
data_component.set_data( data,semantic_models )

tools_component = CRMResultsTools(llm)
tools_component.set_data_component( data_component )
 


In [ ]:
# summarize the CRM model quality
# generate an executive summary report 
# list 3 most supported producers
# which are the 3 best injectors 
# did we run this model using BHP?
# why well XX was discarded 
# which wells were discarded? why 
# is there any evidence of chanelling?
# Evaluate the scenario where the selected producers are shut!
# Show the liquid fit for well XX 
# List the wells with the best liquid fit 
# which wells had a good liquid fit and a good watercut fit?
# Run PFM for those wells 
#  

# which wells have the highest watercut
# identify optimization opportunities 



from langchain.agents import create_agent

 
payload = {'query': "summarize the CRM model quality",
'project_name':'any', 'study_name':'any'
}

llm   = 
agent = create_agent()
 

